In [4]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))

In [8]:
!pip install transformers datasets evaluate jiwer -q

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from datasets import load_dataset, Audio, get_dataset_config_names, get_dataset_split_names
from dataclasses import dataclass
from evaluate import load as load_metric
import os, torch

# CPU Fallback
'''os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
'''

# Load base facebook model to fine-tune
ssl_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-xlsr-53")

# Load pretrained characters from basque fine-tune for token characters
processor = Wav2Vec2Processor.from_pretrained("stefan-it/wav2vec2-large-xlsr-53-basque")
print("\nBasque token vocabulary:")
print(processor.tokenizer.get_vocab())

def preprocess(sample):
    '''
    Preprocess: takes a sample, separates the audio, sampling rate, input and labels
    Args: sample
    Returns: dict with input values and labels
    '''
    audio = sample["audio"]["array"]        # raw waveform
    sr = sample["audio"]["sampling_rate"]   # sample rate

    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
    labels = processor.tokenizer(sample["sentence"].lower()).input_ids

    return {"input_values": inputs.input_values, "labels": labels}

# Load MCV Basque for train split
cv_train_10h = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(7000)
cv_train_10h = cv_train_10h.map(preprocess)

# Validation set
cv_dev = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="dev_cv", streaming=True)
cv_dev = cv_dev.map(preprocess)

'''print(preprocessed_audio.keys())

print("\nSample shape:")
print(preprocessed_audio["input_values"].shape)

print("\nSample labels:")
print(preprocessed_audio["labels"])

print("\nSample labels:")
print(processor.tokenizer.convert_ids_to_tokens(preprocessed_audio["labels"][:5]))'''

@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor

    def __call__(self, features):
        # Pad input_values
        input_features = [{"input_values": f["input_values"].squeeze()} for f in features]
        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")

        # Pad labels with PAD token
        label_features = [f["labels"] for f in features]
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": label_features}, padding=True, return_tensors="pt"
        )
        # Replace PAD token id with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )
        batch["labels"] = labels
        return batch

data_collator = CTCDataCollator(processor=processor)

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/wav2vec2-basque-10h",
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    fp16=False,
    bf16=True,
    learning_rate=1e-4,
    warmup_steps=500,
    save_steps=400,
    logging_steps=400,
    max_steps=50,  # 7000 / 2 batch size * 3 epochs
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id="alexdimmock/wav2vec2-basque-10h",
    hub_strategy="checkpoint",
    hub_token=userdata.get('HF_TOKEN')
    )

wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions.argmax(-1)
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

trainer = Trainer(
    model=ssl_model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=cv_train_10h,
    eval_dataset=cv_dev,
    processing_class=processor.feature_extractor
)

trainer.train()

'''# Exit file
os._exit(0)'''

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Basque token vocabulary:
{'i': 0, 't': 1, 'b': 2, 'n': 3, 'q': 4, 'a': 5, 'd': 6, 'o': 7, 'r': 8, 'h': 9, 'x': 10, 'y': 11, 'ñ': 12, 'f': 14, 'í': 15, 'e': 16, 'z': 17, 'g': 18, 'j': 19, 'v': 20, 'p': 21, 'l': 22, 'm': 23, 's': 24, 'c': 25, 'w': 26, 'k': 27, 'u': 28, '|': 13, '[UNK]': 29, '[PAD]': 30, '<s>': 31, '</s>': 32}


Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,No log,6354.211426,1.005688


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'# Exit file\nos._exit(0)'